# European Option Pricing with Monte Carlo

In this notebook, we use simulated stock prices to estimate the price of European call and put options.

The Monte Carlo idea is:

1. simulate many possible future stock prices;
2. compute the option payoff in each scenario;
3. average the payoffs;
4. discount the average payoff back to today.

The discounted average payoff is the Monte Carlo estimate of the option price.

In [5]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT))

import numpy as np

from src.simulation import simulate_gbm_paths
from src.options import price_european_option_mc

## Simulation parameters

We choose the parameters for the stock price model and for the option contract.

- `S0` is the initial stock price.
- `K` is the strike price.
- `r` is the risk-free rate.
- `sigma` is the volatility.
- `T` is the time to maturity.
- `n_steps` is the number of time steps.
- `n_paths` is the number of simulated futures.

In [6]:
S0 = 100
K = 110
r = 0.05
sigma = 0.20
T = 1.0
n_steps = 252
n_paths = 100_000

paths = simulate_gbm_paths(
    S0=S0,
    r=r,
    sigma=sigma,
    T=T,
    n_steps=n_steps,
    n_paths=n_paths,
    seed=42,
)

terminal_prices = paths[:, -1]

terminal_prices[:10]

array([ 88.37592723, 110.66645157,  76.34895339, 105.01675911,
        97.90382084, 107.5268645 ,  61.51819735,  64.63976533,
        96.09011419, 118.44580651])

## Monte Carlo pricing

We now compute the Monte Carlo price for a European call and a European put.

The function returns:

- the estimated option price;
- the standard error;
- a 95% confidence interval;
- the average undiscounted payoff.

In [7]:
call_result = price_european_option_mc(
    terminal_prices=terminal_prices,
    K=K,
    r=r,
    T=T,
    option_type="call",
)

put_result = price_european_option_mc(
    terminal_prices=terminal_prices,
    K=K,
    r=r,
    T=T,
    option_type="put",
)

call_result, put_result

({'price': 6.07031557794696,
  'standard_error': 0.037198089495226176,
  'confidence_interval': (5.997407322536317, 6.143223833357604),
  'average_payoff': 6.381547312976759},
 {'price': 10.667387059343909,
  'standard_error': 0.03790334797557323,
  'confidence_interval': (10.593096497311786, 10.741677621376033),
  'average_payoff': 11.21431568934388})

In [8]:
print("European call option")
print("--------------------")
print(f"Average payoff: {call_result['average_payoff']:.4f}")
print(f"Monte Carlo price: {call_result['price']:.4f}")
print(f"Standard error: {call_result['standard_error']:.4f}")
print(f"95% confidence interval: {call_result['confidence_interval']}")

print()

print("European put option")
print("-------------------")
print(f"Average payoff: {put_result['average_payoff']:.4f}")
print(f"Monte Carlo price: {put_result['price']:.4f}")
print(f"Standard error: {put_result['standard_error']:.4f}")
print(f"95% confidence interval: {put_result['confidence_interval']}")

European call option
--------------------
Average payoff: 6.3815
Monte Carlo price: 6.0703
Standard error: 0.0372
95% confidence interval: (5.997407322536317, 6.143223833357604)

European put option
-------------------
Average payoff: 11.2143
Monte Carlo price: 10.6674
Standard error: 0.0379
95% confidence interval: (10.593096497311786, 10.741677621376033)
